# PyTestLab Example: Instrument Control Workflow

<div style="background: linear-gradient(135deg, #5333ed 0%, #04e2dc 100%); padding: 1.5rem; border-radius: 12px; margin: 1rem 0; color: white;">
    <h2 style="margin: 0 0 0.5rem 0; color: white;">✨ PyTestLab Professional Notebook</h2>
    <p style="margin: 0; opacity: 0.9; font-size: 1.1rem;">Practical example demonstrating PyTestLab capabilities.</p>
</div>

---

## 📋 Overview

This notebook demonstrates basic instrument control using PyTestLab with a Keysight digital multimeter and arbitrary waveform generator. You'll learn how to:

- Connect to instruments using configuration profiles
- Configure instrument settings programmatically
- Configure a simulated generator and take an independent DMM reading
- Follow best practices for instrument cleanup

> **Note:** Both instruments use independent simulation fixtures. The DMM reading is not a measurement of the generated waveform. Hardware setups need verified addresses, wiring and limits; do not just remove `simulate=True`.

### 🎯 Learning Objectives

- Master basic PyTestLab instrument connections
- Understand instrument configuration workflows
- Learn proper resource management techniques
- Explore DMM and AWG control patterns

### 📚 Prerequisites

- Python 3.11 or newer with PyTestLab installed (`python -m pip install pytestlab`)
- Basic understanding of test instrumentation

---

In [ ]:
# Standard library imports
import os
import sys
from contextlib import ExitStack

from pytestlab import AutoInstrument

print("🚀 PyTestLab environment initialized successfully!")
print(f"📍 Working directory: {os.getcwd()}")
print(f"🐍 Python version: {sys.version.split()[0]}")

## 1. Instrument Connection

This example uses two independent simulation profiles: a Keysight EDU33212A
waveform generator and a Keysight 34470A multimeter. No hardware or VISA
driver is required. The multimeter is not measuring the generator's output.

In [ ]:
resources = ExitStack()
awg = AutoInstrument.from_preset("keysight/EDU33212A", simulate=True)
resources.callback(awg.close)
dmm = AutoInstrument.from_preset("keysight/34470A", simulate=True)
resources.callback(dmm.close)
print("Simulation profiles loaded. Backends open on first use.")

## 2. Instrument Identification

Let's verify our connections by querying instrument identification.

---

In [ ]:
print(f"Multimeter ID: {dmm.id()}")

In [ ]:
# Query waveform generator identification
awg_id = awg.id()
print(f"🌊 AWG ID: {awg_id}")

## 3. Waveform Generator Configuration

Configure the AWG to generate a sine wave output using the chainable facade API.

---

In [ ]:
# Reset AWG to known state and configure basic waveform
awg.reset()
print("🔄 AWG reset to default state")

# Use the facade API for clean, chainable configuration
# This sets up a sine wave and enables the output in one chain
frequency = 25e3  # 25 kHz
amplitude = 1.8  # Volts peak-to-peak

awg.channel(1).setup_sine(frequency=frequency, amplitude=amplitude).enable()
print("🌊 Channel 1 configured:")
print("   Waveform: Sine wave")
print(f"   Frequency: {frequency / 1000:.1f} kHz")
print(f"   Amplitude: {amplitude}V p-p")
print("   Output: Enabled")

## 4. Make a Simulated DC Measurement

Measure the multimeter’s own fixture input. Its result is independent of the generator settings.

In [ ]:
reading = dmm.measure("dc_voltage", range_val=10, uncertainty=False)
print(f"Simulated DC voltage: {float(reading):g} {reading.units}")

## 5. Measurement Summary

Let's summarize our configuration before cleanup.

---

In [ ]:
print(f"Simulated generator set to {frequency / 1000:g} kHz and {amplitude:g} Vpp")
print(f"Independent multimeter fixture: {float(reading):g} {reading.units}")

## 6. Resource Cleanup

Always properly close instrument connections when finished. This is essential for good practice.

---

In [ ]:
try:
    awg.channel(1).disable()
finally:
    resources.close()
print("Closed simulated instruments.")

## Conclusion

You used the same profile-backed API to configure a simulated generator and make an independent simulated multimeter reading.